# 03 - Export ONNX and golden parity fixtures (E01-11, #29)
Exports the **candidate** chart model (`table_features[batch,32] -> chart_scores[batch,10]`, class order = contract), verifies it against scikit-learn
within atol=rtol=1e-6, and builds >=250 golden fixtures (10 classes, edge cases, rejections) for the Python/TypeScript/.NET SDKs.
The role model is **not** exported: there is no role supervision yet. The artifact is a candidate trained on synthetic data under unconfirmed contract readings - not a final model.

In [ ]:
import os, sys, json, subprocess, pathlib, time
MODE = os.environ.get("SQLTOVIZ_MODE", "smoke")            # "smoke" (minutes, subset) | "full"
SEED = 20261127
COMMIT = os.environ.get("SQLTOVIZ_COMMIT", "development")   # pin a commit SHA for publishable runs
REPO = os.environ.get("SQLTOVIZ_REPO")
if not REPO:                                                # Kaggle: clone the repository at the pinned commit
    REPO = "/kaggle/working/sqltoviz"
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "https://github.com/jmmana/sqltoviz", REPO], check=True)
        subprocess.run(["git", "-C", REPO, "checkout", COMMIT], check=True)
sys.path.insert(0, f"{REPO}/research/kaggle")
OUT = os.environ.get("SQLTOVIZ_OUT", "/kaggle/working/out"); os.makedirs(OUT, exist_ok=True)
DATA = os.environ.get("SQLTOVIZ_DATA", "/kaggle/working/data")
GIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("mode", MODE, "| repo commit", GIT, "| out", OUT)
from sqltoviz_data import features as F
F.load_contract()                                           # raises if the contract bytes differ from the pinned SHA256
print(F.CONTRACT_VERSION, F.CONTRACT_SHA256)

In [ ]:
import numpy as np, pathlib
from sqltoviz_data import pipeline as P, splits as S, onnx_export as OX, golden as GD
for d in ("model", "parity", "reports"): os.makedirs(f"{OUT}/{d}", exist_ok=True)
syn = [json.loads(l) for l in open(f"{OUT}/apache-2.0/synthetic.jsonl")]
fs = P.load_features(f"{OUT}/apache-2.0/features_synthetic.npz")
byid = {it["item_id"]: it for it in syn}; syn = [byid[i] for i in fs["item_ids"]]; X = fs["table"]
asg = S.grouped_stratified_split(syn, group=lambda x: x["db_id"], seed=0)
tri = [k for k, i in enumerate(syn) if asg[i["item_id"]] == "train"]; tei = [k for k, i in enumerate(syn) if asg[i["item_id"]] == "test"]
model = OX.fit_contract_model(X[tri], [syn[k]["label"] for k in tri], seed=0, n_estimators=100 if MODE == "full" else 30)
path = OX.export_chart_model(model, f"{OUT}/model/chart_model_candidate.onnx")
sess = OX.session(path)
ver = OX.verify_against_sklearn(model, sess, X[tei])
print(ver); assert ver["within_tolerance"] and ver["argmax_equal_rate"] == 1.0
print("onnx bytes", pathlib.Path(path).stat().st_size, "sha256", OX.sha256_file(path))

## Golden parity fixtures (synthetic + hand-crafted only, redistributable under Apache-2.0)

In [ ]:
cases = GD.handcrafted_cases() + GD.synthetic_cases(syn, per_class=25, max_rows=80, seed=1)
g = GD.build_golden(cases, sess)
rep = GD.verify_golden(json.loads(json.dumps(g)), sess)
print({k: rep[k] for k in ("n_cases", "accepted_ok", "rejected_ok", "max_abs_diff")}, "failures:", rep["failures"])
assert rep["failures"] == [] and g["n_cases"] >= 250, g["n_cases"]
from collections import Counter
print(Counter(c.get("generator_label", c["kind"]) for c in g["cases"]))
json.dump(g, open(f"{OUT}/parity/golden_v1.json", "w"))

## Model card (what the artifact is and is not)

In [ ]:
card = {"status": "CANDIDATE - not final", "contract_version": F.CONTRACT_VERSION, "contract_sha256": F.CONTRACT_SHA256, "repo_commit": GIT,
        "onnx_sha256": OX.sha256_file(path), "class_order": list(OX.CLASSES), "trained_on": "synthetic only (label by construction)",
        "classifier": "RandomForest (fixed defaults, no tuning)", "onnx_vs_sklearn": ver, "golden_cases": g["n_cases"], "golden_verification": {k: rep[k] for k in ("accepted_ok", "rejected_ok", "max_abs_diff")},
        "limitations": ["synthetic labels encode generator conventions", "table-level features cannot separate bar/horizontal_bar and line/area",
                        "contract readings pending review", "role model not trained (no role labels)", "cross-SDK parity not yet measured"]}
json.dump(card, open(f"{OUT}/model/MODEL_CARD.json", "w"), indent=1)
files = {"model/chart_model_candidate.onnx": "apache-2.0", "model/MODEL_CARD.json": "apache-2.0", "parity/golden_v1.json": "apache-2.0"}
man = P.write_manifest(OUT, files, {"mode": MODE, "repo_commit": GIT, "stage": "export"})
print(json.dumps({k: v["sha256"][:12] for k, v in man["files"].items()}, indent=1))